In [2]:
import os
import torch
import torchaudio
import numpy as np
import pickle
import ast
import math
import random
import logging
import gc
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset
from torch.optim import Adam
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Function
from einops.layers.torch import Rearrange
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
from sklearn.metrics import f1_score

In [3]:
# ----------------------------------------------------
# 1. SETUP PATHS FOR KAGGLE DATASET
# ----------------------------------------------------
DATASET_ROOT = "/kaggle/input/datasets/ankit281/kaggle-f0-input/kaggle-input-f0"

train_datasets = {"ESD": os.path.join(DATASET_ROOT, "train")}
val_datasets = {"ESD": os.path.join(DATASET_ROOT, "val")}

train_tokens_orig = {"ESD": os.path.join(DATASET_ROOT, "train_esd.txt")}
val_tokens_orig = {"ESD": os.path.join(DATASET_ROOT, "val_esd.txt")}

f0_file = os.path.join(DATASET_ROOT, "f0.pickle")
ease_embeddings_dir = os.path.join(DATASET_ROOT, "EASE_embeddings")

hparams = {
    "epochs": 200,
    "seed": 1234,
    "n_mel_channels": 80,
    "output_classes": 5,
    "encoder_embedding_dim": 128,
    "speaker_embedding_dim": 192,
    "emotion_embedding_dim": 128,
    "feed_back_last": True,
    "n_frames_per_step_decoder": 1,
    "decoder_rnn_dim": 512,
    "prenet_dim": [256, 256],
    "max_decoder_steps": 2000,
    "stop_threshold": 0.5,
    "attention_rnn_dim": 512,
    "attention_dim": 128,
    "attention_location_n_filters": 32,
    "attention_location_kernel_size": 17,
    "postnet_n_convolutions": 5,
    "postnet_dim": 512,
    "postnet_kernel_size": 5,
    "postnet_dropout": 0.1,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "grad_clip_thresh": 5.0,
    "batch_size": 32,
    "warmup": 7,
    "decay_rate": 0.5,
    "decay_every": 7
}

# Logger setup
logging.basicConfig(
    format="%(asctime)s - %(levelname)s - %(name)s - %(message)s",
    datefmt="%m/%d/%Y %H:%M:%S",
    level=logging.INFO,
)
logger = logging.getLogger(__name__)

SEED = 1234
np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [4]:
# ----------------------------------------------------
# 2. DATASET DEFINITION WITH KAGGLE FIXES
# ----------------------------------------------------
class MyDataset(Dataset):
    def __init__(self, folder, token_file):
        self.folder = folder
        wav_files = os.listdir(folder)
        wav_files = [x for x in wav_files if ".wav" in x]
        self.wav_files = wav_files
        self.sr = 16000
        self.tokens = {}
        
        print(f"Loading F0 statistics from: {f0_file}")
        with open(f0_file, 'rb') as handle:
            self.f0_feat = pickle.load(handle)
            
        print(f"Loading tokens from: {token_file}")
        with open(token_file) as f:
            lines = f.readlines()
            for l in lines:
                d = ast.literal_eval(l)
                name, tokens = d["audio"], d["hubert"]
                tokens_l = tokens.split(" ")
                
                # Cross-platform fix for path separators
                filename = os.path.basename(name.replace("\\", "/"))
                self.tokens[filename] = np.array(tokens_l).astype(int)

    def __len__(self):
        return len(self.wav_files) 

    def getemolabel(self, file_name):
        file_name = int(file_name[5:-4])
        if file_name <= 350:
            return 0
        elif file_name > 350 and file_name <= 700:
            return 1
        elif file_name > 700 and file_name <= 1050:
            return 2
        elif file_name > 1050 and file_name <= 1400:
            return 3
        else:
            return 4

    def getspkrlabel(self, file_name):
        spkr_name = file_name[:4]
        speaker_dict = {}
        for ind in range(11, 21):
            speaker_dict["00"+str(ind)] = ind-11
            
        # Pointing dynamically to the correct dataset ease directory
        npy_filename = file_name.replace(".wav", ".npy")
        speaker_feature = np.load(os.path.join(ease_embeddings_dir, npy_filename))

        return speaker_feature, speaker_dict[spkr_name]
        
    def __getitem__(self, audio_ind): 
        wav_name = self.wav_files[audio_ind]
        class_id = self.getemolabel(wav_name)  
        audio_path = os.path.join(self.folder, wav_name)
        
        (sig, sr) = torchaudio.load(audio_path)
        sig = sig.numpy()[0, :]
        
        tokens = self.tokens[wav_name]
        speaker_feat, speaker_label = self.getspkrlabel(wav_name)
        
        final_sig = sig
        f0 = self.f0_feat[wav_name]

        return final_sig, f0, tokens, class_id, speaker_feat, speaker_label, wav_name

In [5]:
# ----------------------------------------------------
# 3. NETWORK DEFINITIONS
# ----------------------------------------------------
class ReverseLayerF(Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        output = grad_output.neg() * ctx.alpha
        return output, None

class WAV2VECModel(nn.Module):
    def __init__(self, wav2vec, output_dim, hidden_dim_emo):
        super().__init__()
        self.wav2vec = wav2vec
        embedding_dim = wav2vec.config.to_dict()['hidden_size']
        self.out = nn.Linear(hidden_dim_emo, output_dim)
        self.out_spkr = nn.Linear(hidden_dim_emo, 10)
        self.conv1 = nn.Conv1d(in_channels=embedding_dim, out_channels=hidden_dim_emo, kernel_size=5, padding=2)
        self.conv2 = nn.Conv1d(in_channels=hidden_dim_emo, out_channels=hidden_dim_emo, kernel_size=5, padding=2)
        self.conv3 = nn.Conv1d(in_channels=embedding_dim, out_channels=hidden_dim_emo, kernel_size=5, padding=2)
        self.conv4 = nn.Conv1d(in_channels=hidden_dim_emo, out_channels=hidden_dim_emo, kernel_size=5, padding=2)
        self.relu = nn.ReLU()
        
    def forward(self, aud, alpha):
        aud = aud.squeeze(0)
        hidden_all = list(self.wav2vec(aud).hidden_states)
        embedded = sum(hidden_all)
        embedded = embedded.permute(0, 2, 1)

        emo_embedded = self.relu(self.conv1(embedded))
        emo_embedded = self.relu(self.conv2(emo_embedded))
        emo_embedded = emo_embedded.permute(0, 2, 1)
        emo_hidden = torch.mean(emo_embedded, 1).squeeze(1)

        out_emo = self.out(emo_hidden)

        reverse_feature = ReverseLayerF.apply(embedded, alpha)
        embedded_spkr = self.relu(self.conv3(reverse_feature))
        embedded_spkr = self.relu(self.conv4(embedded_spkr))
        hidden_spkr = torch.mean(embedded_spkr, -1).squeeze(-1)
        output_spkr = self.out_spkr(hidden_spkr)
        
        return out_emo, output_spkr, emo_hidden, emo_embedded

class CrossAttentionModel(nn.Module):
    def __init__(self, hidden_dim_q, hidden_dim_k):
        super().__init__()
        HIDDEN_SIZE = 256
        NUM_ATTENTION_HEADS = 4
        self.inter_dim = HIDDEN_SIZE//NUM_ATTENTION_HEADS
        self.num_heads = NUM_ATTENTION_HEADS
        self.fc_q = nn.Linear(hidden_dim_q, self.inter_dim*self.num_heads)
        self.fc_k = nn.Linear(hidden_dim_k, self.inter_dim*self.num_heads)
        self.fc_v = nn.Linear(hidden_dim_k, self.inter_dim*self.num_heads)

        self.multihead_attn = nn.MultiheadAttention(self.inter_dim*self.num_heads,
                                                    self.num_heads,
                                                    dropout = 0.5,
                                                    bias = True,
                                                    batch_first=True)
                                                                                                           
        self.dropout = nn.Dropout(0.5)
        self.layer_norm = nn.LayerNorm(hidden_dim_q, eps = 1e-6)
        self.layer_norm_1 = nn.LayerNorm(hidden_dim_q, eps = 1e-6)
        self.fc = nn.Linear(self.inter_dim*self.num_heads, hidden_dim_q)
        self.fc_1 = nn.Linear(hidden_dim_q, hidden_dim_q)
        self.relu = nn.ReLU()
    
    def forward(self, query_i, key_i, value_i):
        query = self.fc_q(query_i)
        key = self.fc_k(key_i)
        value = self.fc_v(value_i)
        cross, _ = self.multihead_attn(query, key, value, need_weights = True)
        skip = self.fc(cross)
 
        skip += query_i
        skip = self.relu(skip)
        skip = self.layer_norm(skip)

        new = self.fc_1(skip)
        new += skip
        new = self.relu(new)
        out = self.layer_norm_1(new)
        return out

class PitchModel(nn.Module):
    def __init__(self, hparams):
        super(PitchModel, self).__init__()
        self.processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-large-robust-ft-swbd-300h")
        self.wav2vec = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-large-robust-ft-swbd-300h", output_hidden_states=True).to(device)
        
        # Enable gradient checkpointing to save VRAM while keeping all layers unfrozen
        self.wav2vec.gradient_checkpointing_enable()
        
        self.encoder = WAV2VECModel(self.wav2vec, 5, hparams["emotion_embedding_dim"]).to(device)
        self.embedding = nn.Embedding(101, 128, padding_idx=100).to(device)        
        self.fusion = CrossAttentionModel(128, 128).to(device)
        self.linear_layer = nn.Linear(128, 1).to(device)
        self.leaky = nn.LeakyReLU().to(device)
        self.cnn_reg1 = nn.Conv1d(128, 128, kernel_size=(3,), padding=1).to(device)
        self.cnn_reg2 = nn.Conv1d(128, 1, kernel_size=(1,), padding=0).to(device)
        self.speaker_linear = nn.Linear(128, 128).to(device)

    def forward(self, aud, tokens, speaker, lengths, alpha=1.0):
        hidden = self.embedding(tokens.int())
        inputs = self.processor(aud, sampling_rate=16000, return_tensors="pt")
        emo_out, spkr_out, _, emo_embedded = self.encoder(inputs['input_values'].to(device), alpha)
        speaker_temp = speaker.unsqueeze(1).repeat(1, emo_embedded.shape[1], 1)
        speaker_temp = self.speaker_linear(speaker_temp)
        emo_embedded = emo_embedded + speaker_temp
        pred_pitch = self.fusion(hidden, emo_embedded, emo_embedded)
        pred_pitch = pred_pitch.permute(0, 2, 1)
        pred_pitch = self.cnn_reg2(self.leaky(self.cnn_reg1(pred_pitch)))
        pred_pitch = pred_pitch.squeeze(1)
        mask = torch.arange(hidden.shape[1]).expand(hidden.shape[0], hidden.shape[1]).to(device) < lengths.unsqueeze(1)
        pred_pitch = pred_pitch.masked_fill(~mask, 0.0)
        mask = mask.int()

        return pred_pitch, emo_out, spkr_out, mask

def custom_collate(data):
    new_data = {"audio":[], "mask":[], "labels":[], "hubert":[], "f0":[], "speaker":[], "speaker_label":[], "names":[]}
    max_len_f0, max_len_hubert, max_len_aud = 0, 0, 0
    for ind in range(len(data)):
        max_len_aud = max(data[ind][0].shape[-1], max_len_aud)
        max_len_f0 = max(data[ind][1].shape[-1], max_len_f0)
        max_len_hubert = max(data[ind][2].shape[-1], max_len_hubert)
    for i in range(len(data)):
        final_sig = np.concatenate((data[i][0], np.zeros((max_len_aud-data[i][0].shape[-1]))), -1)
        f0_feat = np.concatenate((data[i][1], np.zeros((max_len_f0-data[i][1].shape[-1]))), -1)
        mask = data[i][2].shape[-1]
        hubert_feat = np.concatenate((data[i][2], 100*np.ones((max_len_f0-data[i][2].shape[-1]))), -1)
        labels = data[i][3]
        speaker_feat = data[i][4]
        speaker_label = data[i][5]
        names = data[i][6]
        new_data["audio"].append(final_sig)
        new_data["f0"].append(f0_feat)
        new_data["mask"].append(torch.tensor(mask))
        new_data["hubert"].append(hubert_feat)
        new_data["labels"].append(labels)
        new_data["speaker"].append(speaker_feat)
        new_data["speaker_label"].append(speaker_label)
        new_data["names"].append(names)

    return new_data

In [6]:
# ----------------------------------------------------
# 4. TRAINING LOOP
# ----------------------------------------------------
def train():
    train_dataset = MyDataset(train_datasets["ESD"], train_tokens_orig["ESD"])
    val_dataset = MyDataset(val_datasets["ESD"], val_tokens_orig["ESD"])

    train_loader = DataLoader(train_dataset, batch_size=hparams["batch_size"], shuffle=True, collate_fn=custom_collate, num_workers=4)
    val_loader = DataLoader(val_dataset, batch_size=hparams["batch_size"], shuffle=False, collate_fn=custom_collate, num_workers=4)

    model = PitchModel(hparams)
    
    # Freeze wav2vec model except the layers specified in 'unfreeze'
    for param in model.wav2vec.parameters():
        param.requires_grad = False
        
    # Unfreeze all 24 layers to match the authors' paper configuration
    unfreeze = [str(i) for i in range(0, 24)]
    for name, param in model.wav2vec.named_parameters():
        for num in unfreeze:
            if str(num) in name and 'conv' not in name:
                param.requires_grad = True

    base_lr = 1e-4
    parameters = list(model.parameters()) 
    optimizer = Adam([{'params':parameters, 'lr':base_lr}])
    final_val_loss = 1e20

    for e in range(50):
        model.train()
        torch.cuda.empty_cache()
        import gc
        gc.collect()
        
        tot_loss = 0.0
        tot_train_samples = 0
        tot_val_samples = 0
        pred_tr, gt_tr = [], []
        pred_val, gt_val = [], []
        
        for i, data in enumerate(tqdm(train_loader)):
            p = float(i + e * len(train_loader)) / 100 / len(train_loader)
            alpha = 2. / (1. + np.exp(-10 * p)) - 1
            inputs = torch.tensor(data["audio"]).float().to(device)
            mask = torch.tensor(data["mask"]).to(device)
            tokens = torch.tensor(data["hubert"]).to(device)
            f0_trg = torch.tensor(data["f0"]).to(device)
            labels = torch.tensor(data["labels"]).to(device)
            speaker_label = torch.tensor(data["speaker_label"]).to(device)
            speaker = torch.tensor(data["speaker"]).float().to(device)

            pitch_pred, emo_out, spkr_out, mask_loss = model(inputs, tokens, speaker, mask, alpha)
            pitch_pred = torch.exp(pitch_pred) - 1
            loss1 = (mask_loss * nn.L1Loss(reduction='none')(pitch_pred, f0_trg.float().detach())).sum()
            loss2 = nn.CrossEntropyLoss(reduction='none')(emo_out, labels).sum()
            loss3 = nn.CrossEntropyLoss(reduction='none')(spkr_out, speaker_label).sum()
            
            cur_train_samples = (mask_loss != 0).sum()
            tot_train_samples += cur_train_samples
            
            loss = loss1 + 1000*loss2 + 1000*loss3
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            tot_loss += loss1.detach().item()
            
            pred = torch.argmax(emo_out, dim=1).detach().cpu().numpy()
            pred_tr.extend(list(pred))
            gt_tr.extend(list(labels.detach().cpu().numpy()))

        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for i, data in enumerate(val_loader):
                inputs = torch.tensor(data["audio"]).float().to(device)
                mask = torch.tensor(data["mask"]).to(device)
                tokens = torch.tensor(data["hubert"]).to(device)
                f0_trg = torch.tensor(data["f0"]).to(device)
                labels = torch.tensor(data["labels"]).to(device)
                speaker = torch.tensor(data["speaker"]).float().to(device)

                pitch_pred, emo_out, spkr_out, mask_loss = model(inputs, tokens, speaker, mask)
                pitch_pred = torch.exp(pitch_pred) - 1
                loss1 = (mask_loss * nn.L1Loss(reduction='none')(pitch_pred, f0_trg.float().detach())).sum()
                cur_val_samples = (mask_loss != 0).sum()
                tot_val_samples += cur_val_samples
                
                val_loss += loss1.detach().item()
                pred = torch.argmax(emo_out, dim=1).detach().cpu().numpy()
                pred_val.extend(list(pred))
                gt_val.extend(list(labels.detach().cpu().numpy()))

        if val_loss < final_val_loss:
            # Save state dict only to prevent parametrization serialization errors
            torch.save(model.state_dict(), 'f0_predictor_state.pth')
            final_val_loss = val_loss

        train_loss = tot_loss/tot_train_samples if tot_train_samples > 0 else 0
        train_f1 = f1_score(gt_tr, pred_tr, average='weighted')
        val_loss_log = val_loss/tot_val_samples if tot_val_samples > 0 else 0
        val_f1 = f1_score(gt_val, pred_val, average='weighted')
        e_log = e + 1
        print(f"Epoch {e_log} -> Train Loss: {train_loss:.4f}, Train F1: {train_f1:.4f} | Val Loss: {val_loss_log:.4f}, Val F1: {val_f1:.4f}")

In [ ]:
if __name__ == "__main__":
    train()

Loading F0 statistics from: /kaggle/input/datasets/ankit281/kaggle-f0-input/kaggle-input-f0/f0.pickle
Loading tokens from: /kaggle/input/datasets/ankit281/kaggle-f0-input/kaggle-input-f0/train_esd.txt


/tmp/ipykernel_58/4224290501.py:15: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  self.f0_feat = pickle.load(handle)


Loading F0 statistics from: /kaggle/input/datasets/ankit281/kaggle-f0-input/kaggle-input-f0/f0.pickle
Loading tokens from: /kaggle/input/datasets/ankit281/kaggle-f0-input/kaggle-input-f0/val_esd.txt


06/23/2026 11:02:44 - INFO - httpx - HTTP Request: GET https://huggingface.co/api/models/facebook/wav2vec2-large-robust-ft-swbd-300h/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
06/23/2026 11:02:44 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/facebook/wav2vec2-large-robust-ft-swbd-300h/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
06/23/2026 11:02:44 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/facebook/wav2vec2-large-robust-ft-swbd-300h/resolve/main/chat_template.json "HTTP/1.1 404 Not Found"
06/23/2026 11:02:44 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/facebook/wav2vec2-large-robust-ft-swbd-300h/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
06/23/2026 11:02:44 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/facebook/wav2vec2-large-robust-ft-swbd-300h/resolve/main/audio_tokenizer_config.json "HTTP/1.1 404 Not Found"
06/23/2026 11:02:44 - WARNING - huggingface_

preprocessor_config.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

...


  0%|          | 0/11 [00:00<?, ?it/s]/tmp/ipykernel_58/2789380680.py:44: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  inputs = torch.tensor(data["audio"]).float().to(device)

100%|██████████| 11/11 [00:58<00:00,  5.34s/it]


Epoch 1 -> Train Loss: 60.4411, Train F1: 0.7944 | Val Loss: 103.7492, Val F1: 0.0667


100%|██████████| 11/11 [01:02<00:00,  5.71s/it]


Epoch 2 -> Train Loss: 60.1206, Train F1: 0.8069 | Val Loss: 103.2044, Val F1: 0.0667


100%|██████████| 11/11 [01:03<00:00,  5.74s/it]
